
# Assignment 1: Boolean Model, TF-IDF, and Data retrieval vs. Information retrieval Conceptual Questions

**Student names**: Daniel Høyland <br>
**Group number**: 4 <br>
**Date**: 12.09.2025

## Important notes
Please carefully read the following notes and consider them for the assignment delivery. Submissions that do not fulfill these requirements will not be assessed and should be submitted again.
1. You may work in groups of maximum 2 students.
2. The assignment must be delivered in ipynb format.
3. The assignment must be typed. Handwritten assignments are not accepted.

**Due date**: 14.09.2025 23:59

In this assignment, you will:
- Implement a Boolean retrieval model
- Compute TF-IDF vectors for documents
- Run retrieval on queries
- Answer conceptual questions 

---
## Dataset

You will use the **Cranfield** dataset, provided in this file:

- `cran.all.1400`: The document collection (1400 documents)

**The code to parse the file is ready — just update the cran file path to match your own file location. Use the docs variable in your code for the parsed file**

### Load and parse documents (provided)

Run the cell to parse the Cranfield documents. Update the path so it points to your `cran.all.1400` file.


In [1]:

# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"cran.all.1400"

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)



Parsed 1400 documents.


## 1.1 – Boolean retrieval Model

### 1.1.1 Tokenize documents

Implement tokenization using the given list of stopwords. Create a list of normalized terms per document (e.g., lowercase, remove punctuation/digits; drop stopwords). Store the token lists to use in later steps.

In [2]:
# TODO: Implement tokenization using the given list of stopwords, create list of terms per document

STOPWORDS = set("""a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does doesn't doing don't down
during each few for from further had hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out over own same shan't she
she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll we're
we've were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't
you you'd you'll you're you've your yours yourself yourselves""".split())

# Your code here
tokenized_docs = {}

for doc_id, doc in docs.items():
    # COmbining the title and abstract
    text = (doc["title"] + " " + doc["abstract"]).lower()

    # Keeping letters and spaces
    cleanedText = ""
    for ch in text:
        if ch.isalpha() or ch.isspace():
            cleanedText += ch
        else:
            cleanedText += " "
            
    #Splitting into words based on spaces 
    tokens = cleanedText.split()

    # Removing the stopwords
    filtered = [t for t in tokens if t not in STOPWORDS]

    #Storing the tokenized words back into variables
    tokenized_docs[doc_id] = filtered
    docs[doc_id]["tokens"] = filtered
    
#Testing to see if it works
print(f"Tokenized {len(tokenized_docs)} documents.")
print(docs[1]["tokens"])  

Tokenized 1400 documents.
['experimental', 'investigation', 'aerodynamics', 'wing', 'slipstream', 'experimental', 'investigation', 'aerodynamics', 'wing', 'slipstream', 'experimental', 'study', 'wing', 'propeller', 'slipstream', 'made', 'order', 'determine', 'spanwise', 'distribution', 'lift', 'increase', 'due', 'slipstream', 'different', 'angles', 'attack', 'wing', 'different', 'free', 'stream', 'slipstream', 'velocity', 'ratios', 'results', 'intended', 'part', 'evaluation', 'basis', 'different', 'theoretical', 'treatments', 'problem', 'comparative', 'span', 'loading', 'curves', 'together', 'supporting', 'evidence', 'showed', 'substantial', 'part', 'lift', 'increment', 'produced', 'slipstream', 'due', 'destalling', 'boundary', 'layer', 'control', 'effect', 'integrated', 'remaining', 'lift', 'increment', 'subtracting', 'destalling', 'lift', 'found', 'agree', 'well', 'potential', 'flow', 'theory', 'empirical', 'evaluation', 'destalling', 'effects', 'made', 'specific', 'configuration', '

### Build vocabulary

Create a set (or list) of unique terms from all tokenized documents. Report the number of unique terms.


In [3]:
# TODO: Create a set or list of unique terms

# Report: 
# - Number of unique terms

# Your code here
# Created a set for unique terms
unTerms = set()
# Going throung the documents
for doc in docs.values():
    unTerms.update(doc["tokens"]) #Updating the unique terms

# Displaying the number of unqiue terms
print(f"Unique terms: {len(unTerms)}")

Unique terms: 6934


### Build inverted index

For each term, store the list (or set) of document IDs where the term appears.


In [4]:

# TODO: For each term, store list of document IDs where the term appears
# Your code here
# Inverted indez list
inverted_index = {}
# Going throung the documents
for doc_id, doc in docs.items():
    for term in set(doc["tokens"]): # Gloop for the terms
        if term not in inverted_index: 
            inverted_index[term] = set() # adding a set in the place of the term
        inverted_index[term].add(doc_id) # adding the document ID to the term location

print(f"Inverted index has {len(inverted_index)} terms.")
for term in sorted(inverted_index)[:10]:
    print(term, "->", list(inverted_index[term])[:10])

Inverted index has 6934 terms.
ab -> [744]
abbreviated -> [122]
ability -> [738, 51, 77]
ablated -> [1065]
ablating -> [553, 1098, 1100, 1241]
ablation -> [1096, 553, 1065, 587, 1097, 1098, 1099, 1100, 1101, 1226]
ablative -> [536]
able -> [99, 132, 581, 1114, 908, 914, 695, 986, 763]
abrupt -> [576, 588, 1039]
abruptly -> [992, 662, 439]


### Retrieve documents for a Boolean query (AND/OR)

Create a function to retrieve documents for a Boolean query (AND/OR) with query terms.  


In [5]:
# TODO: Create a function for retrieving documents for a Boolean query (AND/OR) with query terms

def boolean_retrieve(query: str):
    # Tokenize query
    tokens = query.split()
    
    # Making query terms lowercase
    processed = []
    for tok in tokens:
        if tok.upper() in ("AND", "OR"):
            processed.append(tok.upper())
        else:
            processed.append(tok.lower())
    
    # first term
    result = inverted_index.get(processed[0], set()).copy()
    
    # Iterate through operators and next terms
    i = 1
    while i < len(processed):
        op = processed[i]
        term = processed[i + 1]
        docs_for_term = inverted_index.get(term, set())
        
        if op == "AND":
            result = result & docs_for_term
        elif op == "OR":
            result = result | docs_for_term
        
        i += 2
    
    return sorted(result)


In [6]:
# Do not change this code
boolean_queries = [
  "gas AND pressure",
  "structural AND aeroelastic AND flight AND high AND speed OR aircraft",
  "heat AND conduction AND composite AND slabs",
  "boundary AND layer AND control",
  "compressible AND flow AND nozzle",
  "combustion AND chamber AND injection",
  "laminar AND turbulent AND transition",
  "fatigue AND crack AND growth",
  "wing AND tip AND vortices",
  "propulsion AND efficiency"
]

In [7]:
# Run Boolean queries in batch, using the function you created
def run_batch_boolean(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = boolean_retrieve(q)
        results[f"Q{i}"] = res
    return results

boolean_results = run_batch_boolean(boolean_queries)
for qid, res in boolean_results.items():
    print(qid, "=>", res[:5])


Q1 => [27, 49, 85, 101, 110]
Q2 => [12, 14, 29, 47, 51]
Q3 => [5, 399]
Q4 => [1, 61, 244, 265, 342]
Q5 => [118, 131]
Q6 => []
Q7 => [7, 9, 80, 89, 96]
Q8 => []
Q9 => [675]
Q10 => [968]


## Part 1.2 – TF-IDF Indexing


$tf_{i,j} = \text{Raw Frequency}$

$idf_t = \log\left(\frac{N}{df_t}\right)$

### Build document–term matrix (TF and IDF weights)

Compute tf and idf using the formulas above and store the weights in a document–term matrix (rows = documents, columns = terms).



In [8]:
# TODO: Calculate the weights for the documents and the terms using tf and idf weighting. Put these values into a document–term matrix (rows = documents, columns = terms).

# Your code here
import math

# Build vocabulary
vocab = sorted(inverted_index.keys())
term_index = {term: idx for idx, term in enumerate(vocab)}

n = len(docs)  # total number of documents

# Precompute IDF for the terms
idf = {}
for term, doc_ids in inverted_index.items():
    df = len(doc_ids)
    idf[term] = math.log(n / df) if df > 0 else 0.0

# Initialize document-term matrix
# Rows = documents, Columns = terms
doc_term_matrix = [[0.0 for _ in range(len(vocab))] for _ in range(n)]

for doc_id, doc in docs.items():
    # Count term frequency
    tf_counts = {}
    for term in doc["tokens"]:
        tf_counts[term] = tf_counts.get(term, 0) + 1
    
    # Fill row 
    for term, count in tf_counts.items():
        j = term_index[term]
        doc_term_matrix[doc_id-1][j] = count * idf[term]

print(f"Document-term matrix built: {n} docs × {len(vocab)} terms.")
print("Document 1 first 200 weights):")
print(doc_term_matrix[0][:200])


Document-term matrix built: 1400 docs × 6934 terms.
Document 1 first 200 weights):
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.132347370510809, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.7784916128

### Build TF–IDF document vectors

From the matrix, build a TF–IDF vector for each document (consider normalization if needed for cosine similarity).


In [9]:

# TODO: Build TF–IDF document vectors from the document–term matrix
# Your code here
doc_vectors = []

for row in doc_term_matrix:
    # Compute L2 norm
    norm = math.sqrt(sum(w*w for w in row))  
    
    if norm > 0:
        # Normalize vector if not zero
        normalized = [w / norm for w in row]
    else:
        # If zero dont touch
        normalized = row[:]  
    
    # Add normalized vector to list 
    doc_vectors.append(normalized)

print(f"Built {len(doc_vectors)} normalized TF–IDF document vectors.")

Built 1400 normalized TF–IDF document vectors.


### Implement cosine similarity

Implement a function to compute cosine similarity scores between a (tokenized) query and all documents.


In [10]:

# TODO: Create a function for calculating the similarity score of all the documents by their relevance to query terms

    # lowecase
    text = query.lower()
    
    # keep letters and space
    cleaned = ""
    for ch in text:
        if ch.isalpha() or ch.isspace():
            cleaned += ch
        else:
            cleaned += " "
    
    # Split into words
    tokens = cleaned.split()
    
    # Remove stopwords again
    tokens = [t for t in tokens if t not in STOPWORDS]

    # Count term frequencies in query 
    tf_counts = {}
    for term in tokens:
        if term in idf:  
            tf_counts[term] = tf_counts.get(term, 0) + 1

    # Initialize query vector
    query_vec = [0.0] * len(vocab)
    
    # Fill query vector with TF–IDF weights (TF * IDF)
    for term, count in tf_counts.items():
        j = term_index[term]  
        query_vec[j] = count * idf[term]

    # Normalize query vector (L2 norm) 
    norm = math.sqrt(sum(w*w for w in query_vec))
    if norm > 0:
        query_vec = [w / norm for w in query_vec]

    # Compute cosine similarity between query vector and each document vector
    scores = []
    for doc_id, doc_vec in enumerate(doc_vectors, start=1):
        # Dot product between query and document vectors
        score = sum(qw * dw for qw, dw in zip(query_vec, doc_vec))
        scores.append((doc_id, score))

    # Sort documents by similarity score, highest score first
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Return doc IDs that are postive
    return [doc_id for doc_id, _ in scores if _ > 0]




IndentationError: unexpected indent (2769810373.py, line 4)

In [ ]:
# Do not change this code
tfidf_queries = [
  "gas pressure",
  "structural aeroelastic flight high speed aircraft",
  "heat conduction composite slabs",
  "boundary layer control",
  "compressible flow nozzle",
  "combustion chamber injection",
  "laminar turbulent transition",
  "fatigue crack growth",
  "wing tip vortices",
  "propulsion efficiency"
]

In [ ]:
# Run TF-IDF queries in batch (print top-5 results for each), using the function you created
def run_batch_tfidf(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = tfidf_retrieve(q)
        results[f"Q{i}"] = res
    return results

tfidf_results = run_batch_tfidf(tfidf_queries)

for qid, res in tfidf_results.items():
    print(qid, "=>", res[:5])



## Part 1.3 – Conceptual Questions

Answer the following questions:

**1. What is the difference between data retrieval and information retrieval?**
Information retrieval is finding material like documents of an unstructured nature like text that satisfies a infpormational need from a large collection. So like if you search up something on the internet basically. While Data retrieval is more of a structured and exact search for data. So like in a database where you can use a query to get exactly All data about Someone called "John Smith". So in IR if we searth for "John Smith" It will try to find documents that can have either john or/and smith in it thats mostly for you the user while DR is for that system only in a structured data set and it has to be spesifcally "John Smith"

**For the following scenarios, which approach would be suitable data retrieval or information retrieval? Explain your reasoning.** <br>
1.a A clerk in pharmacy uses the following query: Medicine_name = Ibuprofen_400mg
Data retrieval since you need to get a specific information. Like which drawer it is in in their location so it has to be a structured data for them to get anything out of it.

1.b A clerk in pharmacy uses the following query: An anti-biotic medicine 
This would be an information retrieval search since it is very unspesified so it would need to search through unstructured data to find the description of different anti-biotic medicine.

1.c Searching for the schedule of a flight using the following query: Flight_ID = ZEFV2
This is very spesifed so data retrieval again. So it could be a database query on like sas website, so they get the spesifed schedule.

1.d Searching an E-commerce website using the following query to find an specific shoe: Brooks Ghost 15
I would say Information retrieval since a lot of searched a user would do on a E-commerce website is not very specific. Even if the user tried to be very specific with "Brooks Ghost 15" The title of the shoe may not be "Brooks Ghost 15" but it can be in the description of it which is unstructred. I just saw the nect question but I see the point. An E-commerce website can implement title search first so when a user searches "Brooks Ghost 15" first it can be a structured search so a data retrieval. I would guess thats the answer to this question. But I would argue it would be an information retrieval search since most users would have typos, not exact wording or be much more generic. So Information retrieval or a combination is my answer.

1.e Searching the same E-commerce website using the following query: Nice running shoes
As explained above its information retrieval or combination but for this specifically its Information retrieval since this would be based on description of an item, so unstructured search.
